# Pensjonsdemografi og pensjonsvolum

Denne notebooken analyserer ferdige **Gold-data** fra Pensjon Lakehouse-pipelinen i **Databricks**.

Rapporten viser:

1. Utvikling i pensjonsandel 55+ over tid
2. Kommuner med høyest andel innbyggere 55+
3. Aldersgruppefordeling for nyeste år
4. Empirisk aldersfordelingskurve for nyeste år, hvis ettårsfilen finnes i Gold-laget
5. Endring i aldersgrupper fra første til siste år
6. Seniorer 55+ relativt til personer i alderen 20–54
7. Næringer med høyest estimert pensjonsvolum
8. Sammenheng mellom antall lønnstakere og månedslønn per næring

Forutsetning:

- Databricks-clusteret har tilgang til storage accounten.
- Ingen nøkler eller secrets ligger i notebooken.
- Gold-dataene er allerede skrevet til ADLS Gen2 av lakehouse-pipelinen.


## 1. Imports, Spark og ADLS-path

I Databricks finnes `spark` normalt allerede. Fallbacken under gjør at cellen også kan kjøres i en vanlig PySpark-session utenfor Databricks.


In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Last inn Gold-data fra ADLS Gen2

Denne cellen erstatter den lokale DuckDB-/`/tmp`-lesingen fra originalnotebooken med Spark-lesing fra `abfss://...`.

`aldersgruppe_fordeling` støtter både gammelt og nytt filnavn:

- `aldersgruppe_fordeling_siste_ar.parquet`
- `aldersgruppe_fordeling.parquet`

Ettårsfilen `aldersfordeling_siste_ar.parquet` er valgfri. Hvis den ikke finnes i Gold-laget, hopper notebooken over akkurat den empiriske alderskurven, men resten av analysen kjører videre.


In [0]:
def read_gold_parquet(filename: str, view_name: str):
    path = f"{BASE_PATH}/{filename}"
    df = spark.read.parquet(path)
    df.createOrReplaceTempView(view_name)
    print(f"✓ {view_name}: {path}")
    return df


def read_gold_parquet_any(filenames: list[str], view_name: str):
    errors = []
    for filename in filenames:
        try:
            return read_gold_parquet(filename, view_name)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise RuntimeError(
        f"Kunne ikke lese {view_name}. Prøvde:\n" + "\n".join(errors)
    )


pensjonsandel_trend = read_gold_parquet(
    "pensjonsandel_trend.parquet",
    "pensjonsandel_trend",
)

top_kommuner = read_gold_parquet(
    "top_kommuner_pensjonsalder.parquet",
    "top_kommuner",
)

naering_pensjonsvolum = read_gold_parquet(
    "naering_pensjonsvolum.parquet",
    "naering_pensjonsvolum",
)

aldersgruppe_fordeling = read_gold_parquet_any(
    [
        "aldersgruppe_fordeling_siste_ar.parquet",
        "aldersgruppe_fordeling.parquet",
    ],
    "aldersgruppe_fordeling",
)

aldersgruppe_trend = read_gold_parquet(
    "aldersgruppe_trend.parquet",
    "aldersgruppe_trend",
)

try:
    aldersfordeling_siste_ar = read_gold_parquet(
        "aldersfordeling_siste_ar.parquet",
        "aldersfordeling_siste_ar",
    )
    has_aldersfordeling_siste_ar = True
except Exception as exc:
    aldersfordeling_siste_ar = None
    has_aldersfordeling_siste_ar = False
    print(
        "ℹ️ Valgfri fil mangler: aldersfordeling_siste_ar.parquet. "
        "Empirisk ettårsalderskurve hoppes over."
    )
    print(str(exc).split("\n")[0])


## 3. Pensjonsandel 55+ over tid

Denne tabellen og figuren viser hvordan andel innbyggere 55+ utvikler seg over tid.


In [0]:
df_trend = pensjonsandel_trend.orderBy("year").toPandas()

# Støtter både gammel kolonne fra 01-notebooken og ny kolonne fra 02-notebooken.
if "snitt_pensjonsandel_pst" not in df_trend.columns and "pensjonsandel_pst" in df_trend.columns:
    df_trend = df_trend.rename(columns={"pensjonsandel_pst": "snitt_pensjonsandel_pst"})

df_trend["year"] = df_trend["year"].astype(int)

display(df_trend[["year", "snitt_pensjonsandel_pst", "total_55_pluss", "total_befolkning"]])


In [0]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_trend["year"],
    df_trend["snitt_pensjonsandel_pst"],
    marker="o",
)

plt.title("Pensjonsandel 55+ over tid")
plt.xlabel("År")
plt.ylabel("Pensjonsandel 55+ (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Kommuner med høyest andel 55+

Her ser vi hvilke kommuner som har størst andel innbyggere i aldersgruppen 55+.


In [0]:
df_kommuner = (
    top_kommuner
    .select(
        "kommune_label",
        "total_befolkning",
        "pension_age_befolkning",
        F.round(F.col("pension_age_share") * 100, 1).alias("andel_55_pluss"),
    )
    .orderBy(F.col("andel_55_pluss").desc())
    .limit(10)
    .toPandas()
)

# Horisontal barplot leses enklest med lavest verdi øverst i DataFrame.
df_kommuner = df_kommuner.sort_values("andel_55_pluss", ascending=True)

display(df_kommuner)


In [0]:
plt.figure(figsize=(10, 6))

plt.barh(
    df_kommuner["kommune_label"],
    df_kommuner["andel_55_pluss"],
)

plt.title("Kommuner med høyest andel innbyggere 55+")
plt.xlabel("Andel 55+ (%)")
plt.ylabel("Kommune")
plt.tight_layout()
plt.show()


## 5. Aldersgruppefordeling nyeste år

Dette er en grov aldersfordeling basert på rapportklare aldersgrupper i Gold-laget.


In [0]:
df_aldersfordeling = (
    aldersgruppe_fordeling
    .select(
        "aldersgruppe",
        "aldersgruppe_sortering",
        "befolkning",
        F.round(F.col("andel") * 100, 1).alias("andel_prosent"),
    )
    .orderBy("aldersgruppe_sortering")
    .toPandas()
)

display(df_aldersfordeling)


In [0]:
plt.figure(figsize=(10, 5))

plt.bar(
    df_aldersfordeling["aldersgruppe"],
    df_aldersfordeling["andel_prosent"],
)

plt.title("Aldersgruppefordeling nyeste år")
plt.xlabel("Aldersgruppe")
plt.ylabel("Andel av befolkningen (%)")
plt.tight_layout()
plt.show()


## 6. Empirisk aldersfordelingskurve for nyeste år

Denne figuren bruker ettårsaldersfordelingen i Gold-laget hvis filen finnes:

```text
gold/aldersfordeling_siste_ar.parquet
```

Den svarer på spørsmålet:

> Hvordan ser befolkningens aldersprofil ut nå?


In [0]:
if has_aldersfordeling_siste_ar:
    df_alder_siste_ar = (
        aldersfordeling_siste_ar
        .select(
            "year",
            "alder",
            "befolkning",
            (F.col("andel") * 100).alias("andel_prosent"),
        )
        .orderBy("alder")
        .toPandas()
    )

    df_alder_siste_ar["glattet_andel_prosent"] = (
        df_alder_siste_ar["andel_prosent"]
        .rolling(window=5, center=True, min_periods=1)
        .mean()
    )

    display(df_alder_siste_ar)
else:
    df_alder_siste_ar = pd.DataFrame()
    print("Hopper over: aldersfordeling_siste_ar.parquet finnes ikke i Gold-laget.")


In [0]:
if not df_alder_siste_ar.empty:
    latest_year_age_curve = int(df_alder_siste_ar["year"].iloc[0])

    plt.figure(figsize=(11, 6))

    plt.plot(
        df_alder_siste_ar["alder"],
        df_alder_siste_ar["glattet_andel_prosent"],
        linewidth=2,
    )

    plt.fill_between(
        df_alder_siste_ar["alder"],
        df_alder_siste_ar["glattet_andel_prosent"],
        alpha=0.2,
    )

    plt.title(f"Aldersfordelingskurve for befolkningen, {latest_year_age_curve}")
    plt.xlabel("Alder")
    plt.ylabel("Andel av befolkningen (%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Ingen ettårsalderskurve å plotte.")


## 7. Endring i aldersgrupper fra første til siste år

Denne figuren viser hvilke aldersgrupper som øker eller faller som andel av befolkningen.

Dette er mer informativt enn å plotte flere nesten like aldersfordelingskurver over tid.


In [0]:
df_aldersgruppe_trend_long = (
    aldersgruppe_trend
    .select(
        "year",
        "aldersgruppe",
        "aldersgruppe_sortering",
        "befolkning",
        F.round(F.col("andel") * 100, 1).alias("andel_prosent"),
    )
    .orderBy("year", "aldersgruppe_sortering")
    .toPandas()
)

first_year = df_aldersgruppe_trend_long["year"].min()
last_year = df_aldersgruppe_trend_long["year"].max()

df_aldersgruppe_endring = (
    df_aldersgruppe_trend_long
    .pivot(
        index=["aldersgruppe", "aldersgruppe_sortering"],
        columns="year",
        values="andel_prosent",
    )
    .reset_index()
)

df_aldersgruppe_endring["endring_prosentpoeng"] = (
    df_aldersgruppe_endring[last_year] - df_aldersgruppe_endring[first_year]
)

df_aldersgruppe_endring = df_aldersgruppe_endring.sort_values("aldersgruppe_sortering")

display(df_aldersgruppe_endring)


In [0]:
plt.figure(figsize=(10, 5))

plt.axhline(0, linewidth=1)

plt.bar(
    df_aldersgruppe_endring["aldersgruppe"],
    df_aldersgruppe_endring["endring_prosentpoeng"],
)

plt.title(f"Endring i aldersgruppenes befolkningsandel fra {first_year} til {last_year}")
plt.xlabel("Aldersgruppe")
plt.ylabel("Endring i prosentpoeng")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Seniorer relativt til yrkesaktiv alder

Denne figuren viser forholdet mellom:

- seniorer 55+
- personer i alderen 20–54

Dette gir en enkel indikator på demografisk pensjonspress.


In [0]:
df_seniorpress = (
    aldersgruppe_trend
    .groupBy("year")
    .agg(
        F.sum(
            F.when(F.col("aldersgruppe").isin("20-34", "35-49", "50-54"), F.col("befolkning"))
            .otherwise(F.lit(0))
        ).alias("yrkesaktiv_20_54"),
        F.sum(
            F.when(F.col("aldersgruppe").isin("55-61", "62-66", "67-74", "75+"), F.col("befolkning"))
            .otherwise(F.lit(0))
        ).alias("senior_55_pluss"),
    )
    .orderBy("year")
    .toPandas()
)

df_seniorpress["senior_per_yrkesaktiv"] = (
    df_seniorpress["senior_55_pluss"]
    / df_seniorpress["yrkesaktiv_20_54"]
)

display(df_seniorpress)


In [0]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_seniorpress["year"],
    df_seniorpress["senior_per_yrkesaktiv"],
    marker="o",
)

plt.title("Seniorer 55+ per person i alderen 20–54")
plt.xlabel("År")
plt.ylabel("Forholdstall")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Næringer med høyest estimert pensjonsvolum

Estimert pensjonsvolum er beregnet i pipelinen som:

```text
lønnstakere × månedslønn × 12 × 0.02
```

`Alle næringer` ekskluderes her fordi det er en totalsum, ikke en enkelt næring.


In [0]:
df_naering = (
    naering_pensjonsvolum
    .filter(F.col("naering_label") != "Alle næringer")
    .select(
        "naering_label",
        "lonsstakere",
        "manedslonn",
        "estimert_pensjonsvolum",
        (F.col("estimert_pensjonsvolum") / F.lit(1_000_000_000)).alias("estimert_pensjonsvolum_mrd"),
    )
    .orderBy(F.col("estimert_pensjonsvolum").desc())
    .limit(10)
    .toPandas()
)

# Horisontal barplot leses enklest med lavest verdi øverst i DataFrame.
df_naering = df_naering.sort_values("estimert_pensjonsvolum_mrd", ascending=True)

display(df_naering)


In [0]:
plt.figure(figsize=(11, 6))

plt.barh(
    df_naering["naering_label"],
    df_naering["estimert_pensjonsvolum_mrd"],
)

plt.title("Næringer med høyest estimert pensjonsvolum")
plt.xlabel("Estimert pensjonsvolum, mrd. kr")
plt.ylabel("Næring")
plt.tight_layout()
plt.show()


## 10. Lønnstakere og månedslønn per næring

Denne figuren gir et ekstra blikk på hva som driver pensjonsvolumet: mange ansatte, høy månedslønn, eller en kombinasjon.


In [0]:
df_naering_scatter = (
    naering_pensjonsvolum
    .filter(F.col("naering_label") != "Alle næringer")
    .select(
        "naering_label",
        "lonsstakere",
        "manedslonn",
        (F.col("estimert_pensjonsvolum") / F.lit(1_000_000_000)).alias("estimert_pensjonsvolum_mrd"),
    )
    .orderBy(F.col("estimert_pensjonsvolum_mrd").desc())
    .toPandas()
)

display(df_naering_scatter)


In [0]:
plt.figure(figsize=(10, 6))

plt.scatter(
    df_naering_scatter["lonsstakere"],
    df_naering_scatter["manedslonn"],
)

plt.title("Lønnstakere og månedslønn per næring")
plt.xlabel("Antall lønnstakere")
plt.ylabel("Månedslønn")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Kort oppsummering

Denne cellen trekker ut noen nøkkelpunkter fra analysen.


In [0]:
import pandas as pd

def to_pandas(df):
    return df.toPandas() if hasattr(df, "toPandas") else df.copy()


trend = to_pandas(df_trend)
kommuner = to_pandas(df_kommuner)
naering = to_pandas(df_naering)
aldersfordeling = to_pandas(df_aldersfordeling)
seniorpress = to_pandas(df_seniorpress)

alder_siste_ar = (
    to_pandas(df_alder_siste_ar)
    if "df_alder_siste_ar" in globals()
    else pd.DataFrame()
)

# Rydd datatyper
trend["year"] = pd.to_numeric(trend["year"], errors="coerce")
seniorpress["year"] = pd.to_numeric(seniorpress["year"], errors="coerce")

# Hent nøkkeltall
latest = trend.dropna(subset=["year"]).sort_values("year").iloc[-1]
latest_year = int(latest["year"])

top_kommune = kommuner.sort_values("andel_55_pluss", ascending=False).iloc[0]
top_naering = naering.sort_values("estimert_pensjonsvolum_mrd", ascending=False).iloc[0]
largest_age_group = aldersfordeling.sort_values("andel_prosent", ascending=False).iloc[0]

oldest = aldersfordeling.loc[aldersfordeling["aldersgruppe"] == "75+"]

seniorpress_match = seniorpress.loc[seniorpress["year"] == latest_year]

if not seniorpress_match.empty:
    seniorpress_row = seniorpress_match.iloc[0]
elif not seniorpress.dropna(subset=["year"]).empty:
    seniorpress_row = seniorpress.dropna(subset=["year"]).sort_values("year").iloc[-1]
else:
    seniorpress_row = None

if not alder_siste_ar.empty:
    peak_age = alder_siste_ar.sort_values("glattet_andel_prosent", ascending=False).iloc[0]
else:
    peak_age = None

# Lag oppsummering
summary_rows = [
    (
        "Siste år i datasettet",
        str(latest_year),
        "Basert på pensjonsandel-trenden",
    ),
    (
        "Pensjonsandel 55+",
        f"{latest['snitt_pensjonsandel_pst']:.1f} %",
        f"Siste observasjon, {latest_year}",
    ),
    (
        "Kommune med høyest andel 55+",
        str(top_kommune["kommune_label"]),
        f"{float(top_kommune['andel_55_pluss']):.1f} %",
    ),
    (
        "Største aldersgruppe",
        str(largest_age_group["aldersgruppe"]),
        f"{float(largest_age_group['andel_prosent']):.1f} % av befolkningen",
    ),
    (
        "Andel 75+",
        f"{float(oldest.iloc[0]['andel_prosent']):.1f} %" if not oldest.empty else "Mangler",
        "Siste tilgjengelige aldersfordeling",
    ),
    (
        "Seniorer 55+ per person 20–54",
        f"{float(seniorpress_row['senior_per_yrkesaktiv']):.2f}" if seniorpress_row is not None else "Mangler",
        f"År {int(seniorpress_row['year'])}" if seniorpress_row is not None else "Seniorpress kunne ikke beregnes",
    ),
    (
        "Næring med høyest estimert pensjonsvolum",
        str(top_naering["naering_label"]),
        f"{float(top_naering['estimert_pensjonsvolum_mrd']):.1f} mrd. kr",
    ),
    (
        "Toppunkt i glattet aldersfordeling",
        f"Alder {int(peak_age['alder'])}" if peak_age is not None else "Mangler",
        f"{float(peak_age['glattet_andel_prosent']):.2f} % av befolkningen"
        if peak_age is not None
        else "Ettårsaldersfordeling er ikke tilgjengelig",
    ),
]

# Vis som Spark DataFrame i Databricks
summary_df = spark.createDataFrame(
    summary_rows,
    ["Nøkkeltall", "Verdi", "Kommentar"],
)

display(summary_df)